# Prompt-injection screening

**Recipe 26** · Level 3 (Intermediate) · Decision type: `Noul`

Flag instruction-redirection attempts in retrieved text using adversarial and benign quotation fixtures to measure false positives and missed attacks.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S04: [TypeSafe AI: How to build with TypeSafe](https://docs.typesafe.ai/concepts/how-to-build-with-system-one)
- S07: [TypeSafe AI: Jev 1.13 jaggedness](https://docs.typesafe.ai/model-jaggedness/jev-1.13)

## What you will build

A screening pass over 48 fabricated retrieved passages — forum posts, support tickets, security blog entries, incident write-ups and training material — covering every combination of the two propositions below: text that only tries to redirect the reader's assistant (direct, indirect, hidden in a code comment, politely phrased, or split across sentences); text that only quotes, reports or discusses such an attempt without issuing one; text that does both at once, reporting a known attack pattern and separately issuing a live one; and ordinary text that does neither, with nothing to do with prompt injection at all. Jev answers two independent `Noul` propositions about the same excerpt in one request: does it contain a redirect instruction, and does it merely discuss one. Python composes the two answers into one of three outcomes — `flag`, `pass` or `review` — in a single tested function, using a business threshold chosen per proposition and one confidence gate shared by both. The evaluation reports a miss rate on the adversarial passages and a false-positive rate on the benign ones, and shows both a case the shared gate cannot catch by construction and a case where the gate catches genuine uncertainty on one proposition while the other reads confidently and correctly — real-fixture evidence that each proposition can change the outcome on its own. Screening like this is one layer of defence, not a guarantee: it can miss an attack and it can misread a quotation as one, and TypeSafe's own notes on Jev 1.13 ([S07](https://docs.typesafe.ai/model-jaggedness/jev-1.13)) describe real limits on how adversarial content is handled, which this recipe does not try to quantify.

## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only. `get_backend(fixtures=...)` replays the 48 stored answers in `fixtures/responses.json` offline; setting `JEV_COOKBOOK_LIVE=1` (see the README) swaps in live calls to the same two question definitions and changes nothing else. `run_header` states which mode ran, and every number below carries that mode with it. Each passage needs exactly one request (both propositions are independent questions over the same excerpt, so they are asked together), so this notebook's request count and its decision count are not the same thing: 48 requests produce 96 label decisions, two per passage.

In [ ]:
from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    confusion_matrix,
    evaluate_outcomes,
    evaluate_selective,
    evaluate_threshold,
    pool_label_decisions,
    select_confidence_threshold,
    select_threshold,
    selective_curve,
    threshold_sweep,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ReviewQueue
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_confusion_matrix,
    plot_risk_coverage,
    plot_threshold_sweep,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path())
questions = helpers.build_questions()
# N for the header is the number of examples the metrics are about: validation and test. The
# two demo passages are shown below but never scored.
scored = [e for e in examples if e.split not in ("train", "demo")]

offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
# CONTRIBUTING.md section 2: a number from the validation split is a selection step, not a
# result. A synthetic or scripted run carries both labels on a validation number.
selection = " (a selection step, not a reported result)"

# Every passage needs at most one request; this cache makes sure a passage shown individually
# and later scored in its split is decided once, so the whole notebook makes exactly one
# request per fixture passage in live mode, never more.
decided = {}


def decide(example):
    if example.id not in decided:
        decided[example.id] = backend.decide(helpers.build_state(example.fields), questions)
    return decided[example.id]


run_header(26, "Prompt-injection screening", backend=backend, n_examples=len(scored))

## The state

The state is everything Jev sees, and Python builds it. Each example in `fixtures/inputs.jsonl` carries `fields`: a `document`, the full fabricated passage. `build_state` passes on an excerpt of it only — `helpers.excerpt` cuts the document at a word boundary once it passes `MAX_EXCERPT_CHARS` (320) and marks the cut with "…" — so a retrieved document's full length is never assumed to matter, the way a real retrieval step would hand over a bounded window rather than an entire file. The example id never passes through the model either; it stays in Python and is attached to the outcome at the end.

In [ ]:
import json

demo = {e.id: e for e in examples if e.split == "demo"}
flag_demo = demo["d01"]
print("Python has:")
print(json.dumps(flag_demo.fields, indent=2))
state = helpers.build_state(flag_demo.fields)
print("Jev sees:")
print(json.dumps(state, indent=2))

`ta05`, one of the `test` passages below, is long enough to show the cut actually happening: Python excerpts it to 320 characters, well before the padding at the end of the message, while the line that makes it adversarial sits early enough to survive.

In [ ]:
long_example = next(e for e in examples if e.id == "ta05")
document = long_example.fields["document"]
excerpted = helpers.excerpt(document)
print(f"document: {len(document)} characters")
print(f"excerpt: {len(excerpted)} characters")
print(excerpted)

## The questions

Two independent propositions describe the same excerpt, so `backend.decide` sends them together in one request: TypeSafe's primitives page ([S02](https://docs.typesafe.ai/primitives)) says independent questions over the same state belong in one request. Each is written as a statement that is true or false, as CONTRIBUTING.md requires of a `Noul`, not as a question: `redirect` is true when the excerpt itself addresses the reader's assistant with an instruction to change or ignore its instructions; `discusses` is true when the excerpt quotes or describes an instruction of that kind without issuing one. They stay two separate `Noul` propositions rather than one two-option `Choice` because they are not mutually exclusive readings Jev must pick between — a passage can in principle read as both at once, or as neither — and because a `Choice` would ask Jev to adjudicate the very distinction this recipe wants two separate, checkable signals about.

In [ ]:
for name, question in questions.items():
    print(name)
    print(" ", question.instructions)
    for side, description in question.criteria.items():
        print(f"    {side}: {description}")

## One answer up close

Before any aggregate, look at six typed answers: a passage that plainly redirects the reader's assistant, one that plainly does not, a softly-worded attack the stored answer gets wrong, a benign passage that quotes an attack closely enough to pull a Noul probability most of the way toward the adversarial reading, a passage that reports a known attack pattern and also issues a live one of its own, and an ordinary passage with nothing to do with prompt injection at all. Each answer holds just `noul`, the probability that its proposition is true, and a `provenance` line; a Noul has no separate confidence field in the API response at all (S02, S03), unlike `Choice` and `Score`. All six come from `demo` or `validation`, never from `test`, which is reported once further down.

In [ ]:
flag_answer = decide(flag_demo)
print(flag_demo.fields["document"])
show_answer(flag_answer["redirect"])
show_answer(flag_answer["discusses"])
plot_answer_probabilities(
    flag_answer["redirect"], title="redirect, a passage that plainly redirects"
)

The passage is a direct instruction aimed at the assistant reading it. The stored `redirect` probability, 0.97, sits near the top of the scale; `discusses`, 0.03, near the bottom. Both readings agree, and agree with the [gold label](../../docs/glossary.md#gold-label) — the correct classification this recipe records ahead of time, from the criteria above, never from the model, so a stored answer can be checked against it after the fact.

In [ ]:
pass_demo = demo["d02"]
pass_answer = decide(pass_demo)
print(pass_demo.fields["document"])
show_answer(pass_answer["redirect"])
show_answer(pass_answer["discusses"])

This passage is a short security note reporting what attackers do, quoting the phrase they tend to use. `redirect` is stored low (0.04) and `discusses` high (0.97): the passage is about a redirect attempt without addressing the assistant as one.

In [ ]:
uncertain_example = next(e for e in examples if e.id == "va08")
uncertain_answer = decide(uncertain_example)
print(uncertain_example.fields["document"])
show_answer(uncertain_answer["redirect"])
show_answer(uncertain_answer["discusses"])

This is the softest-worded attack in the fixtures: an offer, not an instruction. Its stored `redirect` probability, 0.88, sits close enough to this recipe's eventual business threshold that whether it tags `True` or `False` is a near thing, and `noul_confidence`, `|2p - 1|` ([S03](https://docs.typesafe.ai/confidence)), reads it as only moderately confident despite the probability itself looking high — "Python's part" below is where a passage this close to its own threshold, with confidence this unconvincing, gets sent to a person instead of a forced guess.

In [ ]:
lookalike_example = next(e for e in examples if e.id == "vb05")
lookalike_answer = decide(lookalike_example)
print(lookalike_example.fields["document"])
show_answer(lookalike_answer["redirect"])
show_answer(lookalike_answer["discusses"])

An incident write-up that reproduces an attack's exact wording to document it. Quoting the attack almost verbatim pulls the stored `redirect` probability to 0.89 — close to the adversarial cluster, even though the gold label is still `False`: this is the gold label's own hard case, a benign look-alike, not an error in the fixture. `discusses` is stored high (0.95) too, correctly.

In [ ]:
mixed_example = next(e for e in examples if e.id == "vm01")
mixed_answer = decide(mixed_example)
print(mixed_example.fields["document"])
show_answer(mixed_answer["redirect"])
show_answer(mixed_answer["discusses"])

This passage reports a known attack phrase for documentation purposes and then separately issues a live redirect instruction of its own — the "mixed" hard case CONTRIBUTING.md section 5 names, and the reason this recipe asks two independent `Noul` propositions rather than one `Choice`. Both readings come back high: `redirect` 0.95, `discusses` 0.93. "Python's part" below shows why a confident `redirect` tag reports `flag` here even though `discusses` also reads true — a passage can quote one attack while issuing a second, live one, and the live one is what a reader needs to see first.

In [ ]:
neither_example = next(e for e in examples if e.id == "vn01")
neither_answer = decide(neither_example)
print(neither_example.fields["document"])
show_answer(neither_answer["redirect"])
show_answer(neither_answer["discusses"])

This passage is ordinary retrieved content with nothing to do with prompt injection at all — the "no match" hard case CONTRIBUTING.md section 5 also names, and the case a real screener sees most often. Both readings come back low: `redirect` 0.02, `discusses` 0.02. Neither proposition is true, and `screen` below reports `pass` for the plain reason that nothing was found, not because either label happened to clear a threshold.

## Python's part

Jev supplies two probabilities; everything that happens with them is code. `helpers.decide_labels` turns each one into a [business threshold](../../docs/glossary.md#business-threshold) tag — `redirect >= t_redirect`, `discusses >= t_discusses` — and checks it against one shared [confidence gate](../../docs/glossary.md#confidence-gate), `noul_confidence(noul) >= gate`, for either label. `helpers.screen` then composes the two label decisions into one outcome: a passage with either label gated goes to an explicit [review](../../docs/glossary.md#review) outcome, whichever way its tags read; otherwise a confident `redirect` tag reports `flag`; otherwise a confident `discusses` tag reports `pass`, naming the quotation rather than silently agreeing with "nothing found"; otherwise it is `pass` because nothing was found at all. `tests/test_helpers.py` checks every one of those branches, and every label/outcome combination, directly.

Both per-label thresholds are chosen independently, by F1, and the one shared gate is chosen afterwards from both labels' pooled decisions — all three on `validation`, then frozen.

In [ ]:
val_examples = [e for e in examples if e.split == "validation"]
val_answers = {e.id: decide(e) for e in val_examples}
val_redirect = [val_answers[e.id]["redirect"].noul for e in val_examples]
val_discusses = [val_answers[e.id]["discusses"].noul for e in val_examples]
val_gold_redirect = [labels[e.id]["redirect"] for e in val_examples]
val_gold_discusses = [labels[e.id]["discusses"] for e in val_examples]

t_redirect = select_threshold(val_gold_redirect, val_redirect, objective="f1")
t_discusses = select_threshold(val_gold_discusses, val_discusses, objective="f1")
thresholds = {helpers.REDIRECT: t_redirect, helpers.DISCUSSES: t_discusses}
print(f"redirect business threshold, frozen from here on: {t_redirect:.3f}{selection}{check}")
print(f"discusses business threshold, frozen from here on: {t_discusses:.3f}{selection}{check}")

val_ids = [e.id for e in val_examples]
gold_by_label = {helpers.REDIRECT: val_gold_redirect, helpers.DISCUSSES: val_gold_discusses}
noul_by_label = {helpers.REDIRECT: val_redirect, helpers.DISCUSSES: val_discusses}
val_pooled_correct, val_pooled_confidence, val_pooled_keys = pool_label_decisions(
    val_ids, list(helpers.LABELS), gold_by_label, noul_by_label, thresholds
)
gate = select_confidence_threshold(val_pooled_correct, val_pooled_confidence, target_accuracy=1.0)
print(f"confidence gate, frozen from here on: {gate:.3f}{selection}{check}")

for shown_id, shown in (
    ("d01", flag_demo),
    ("d02", pass_demo),
    ("va08", uncertain_example),
    ("vb05", lookalike_example),
    ("vm01", mixed_example),
    ("vn01", neither_example),
):
    answer = decide(shown)
    decisions = helpers.decide_labels(answer, thresholds, gate)
    result = helpers.screen(decisions)
    print(
        f"{shown_id}: redirect {answer['redirect'].noul:.2f}, discusses "
        f"{answer['discusses'].noul:.2f} -> {result.outcome} ({result.reason})"
    )

### Why one shared gate, and its blind spot

A multi-label `Noul` rule's three-path pattern — answer yes, answer no, or send a passage to a person — needs every (passage, label) decision's correctness and confidence pooled before a [coverage](../../docs/glossary.md#coverage) and [risk](../../docs/glossary.md#risk) curve can be drawn over both labels at once; `pool_label_decisions` does that pooling, and `selective_curve` is the sanity check that confidence actually separates right answers from wrong ones on this data, shown next rather than asserted, as a single shared gate over two different propositions. `|2p - 1|` measures distance from an even split, not distance from whichever business threshold a label happens to use: a label whose own threshold sits near 1.0 can read as confidently right even a little below it, because `noul_confidence` is never told where that threshold is. The gate chosen here is structurally blind to exactly that shape of error, on either label, and "Evaluation" below shows one. The curve below is a narrow sanity check: it has exactly one wrong decision among the forty-six pooled (passage, label) decisions on this split, so its accuracy reads 1.0000 at every threshold until the last row, where that one decision enters.

In [ ]:
val_curve = selective_curve(val_pooled_correct, val_pooled_confidence)
print(f"pooled, {len(val_pooled_correct)} label decisions{selection}{check}")
print(
    f"ungated accuracy: {sum(val_pooled_correct) / len(val_pooled_correct):.4f}{selection}{check}"
)
for t, cov, acc, risk in zip(
    val_curve.thresholds, val_curve.coverage, val_curve.accuracy, val_curve.risk, strict=True
):
    marker = " <- frozen" if abs(t - gate) < 1e-9 else ""
    print(
        f"  gate >= {t:.3f}: coverage {cov:.4f}, accuracy {acc:.4f}, "
        f"risk {risk:.4f}{marker}{selection}{check}"
    )
plot_risk_coverage(
    val_curve,
    reference_risk=1 - sum(val_pooled_correct) / len(val_pooled_correct),
    title=f"Validation, pooled label decisions{check}",
)

`jev_cookbook.simulation.ReviewQueue` is the shared primitive for the `review` outcome ([docs/simulation.md](../../docs/simulation.md)): `submit` records an item, the reason it was sent, and the typed answers behind that reason, and nothing in it executes anything. `ReviewQueue` itself keeps submission order, oldest first; it does not reorder anything on its own, so a queue that reads sorted is sorted once, before every item is submitted, by `helpers.review_sort_key` — the lower of the two labels' own confidence, so the pair Python is least sure about surfaces first. A preview on the six passages shown above, with the full item a reviewer would actually see:

In [ ]:
preview_items = []
for shown_id, shown in (
    ("d01", flag_demo),
    ("d02", pass_demo),
    ("va08", uncertain_example),
    ("vb05", lookalike_example),
    ("vm01", mixed_example),
    ("vn01", neither_example),
):
    answer = decide(shown)
    decisions = helpers.decide_labels(answer, thresholds, gate)
    result = helpers.screen(decisions)
    if result.outcome == helpers.REVIEW:
        preview_items.append((shown_id, shown, result))

preview_items.sort(key=lambda row: (helpers.review_sort_key(row[2]), row[0]))
print(f"preview: {len(preview_items)} of these six would be queued{selection}{check}")
for shown_id, shown, result in preview_items:
    item = helpers.queue_item(shown_id, shown.fields, result)
    print(f"  {item['id']}: {result.reason}, sort key {helpers.review_sort_key(result):.3f}")
    print(f"    passage: {item['passage']}")
    print(
        f"    redirect: noul {item[helpers.REDIRECT]['noul']:.2f}, confidence "
        f"{item[helpers.REDIRECT]['confidence']:.4f}"
    )
    print(
        f"    discusses: noul {item[helpers.DISCUSSES]['noul']:.2f}, confidence "
        f"{item[helpers.DISCUSSES]['confidence']:.4f}"
    )

## Evaluation

Both per-label thresholds and the shared confidence gate were chosen and frozen above, on `validation`. This section reports, per question, each label's own precision and recall against its threshold alone (no gate); at task level, the counts `helpers.screen` actually returns and [`evaluate_outcomes`](../../docs/evaluation.md)'s coverage, [accuracy](../../docs/glossary.md#accuracy) and risk over the composed outcome; and the two numbers the use case asks for directly — the miss rate on the adversarial group and the false-positive rate on the benign group — each reported both gated (what the frozen rule reports) and ungated (what the business threshold alone would say with no review outcome at all), so the two can be compared side by side. `test` is used once, after every choice above is frozen.

In [ ]:
val_results = {
    e.id: helpers.screen(helpers.decide_labels(decide(e), thresholds, gate)) for e in val_examples
}
val_counts = {helpers.FLAG: 0, helpers.PASS: 0, helpers.REVIEW: 0}
for e in val_examples:
    val_counts[val_results[e.id].outcome] += 1
val_accepted = [val_results[e.id].outcome != helpers.REVIEW for e in val_examples]
val_correct = [val_results[e.id].outcome == labels[e.id]["outcome"] for e in val_examples]
val_outcomes = evaluate_outcomes(val_accepted, val_correct)

print(f"validation, {len(val_examples)} passages{selection}{check}")
print(
    f"flag: {val_counts[helpers.FLAG]}  pass: {val_counts[helpers.PASS]}  "
    f"review: {val_counts[helpers.REVIEW]}{selection}{check}"
)
print(f"coverage: {val_outcomes.coverage:.4f}{selection}{check}")
print(f"accuracy among answered: {val_outcomes.accuracy:.4f}{selection}{check}")
print(f"risk among answered: {val_outcomes.risk:.4f}{selection}{check}")

[`precision`](../../docs/glossary.md#precision) is the fraction of passages tagged `redirect` that actually redirect; [`recall`](../../docs/glossary.md#recall) is the fraction of real redirect attempts in this split the rule tags at all; [`f1`](../../docs/glossary.md#f1) is their harmonic mean. The sweep below is `redirect`'s own business rule alone, over every threshold `select_threshold` considered on `validation`, printed and then drawn.

In [ ]:
sweep = threshold_sweep(val_gold_redirect, val_redirect)
for point in sweep:
    print(
        f"  threshold {point.threshold:.3f}: precision {point.precision:.4f}, "
        f"recall {point.recall:.4f}, f1 {point.f1:.4f}{selection}{check}"
    )
plot_threshold_sweep(
    sweep, chosen=t_redirect, title=f"Validation, redirect, {len(val_examples)} passages{check}"
)

On `test`, each label's own business rule, with no gate at all, is reported first with `evaluate_threshold`; then the composed, task-level outcome `helpers.screen` actually returns, and its coverage, accuracy and risk from `evaluate_outcomes`, exactly as for `validation` above.

In [ ]:
test_examples = [e for e in examples if e.split == "test"]
test_answers = {e.id: decide(e) for e in test_examples}
test_redirect = [test_answers[e.id]["redirect"].noul for e in test_examples]
test_discusses = [test_answers[e.id]["discusses"].noul for e in test_examples]
test_gold_redirect = [labels[e.id]["redirect"] for e in test_examples]
test_gold_discusses = [labels[e.id]["discusses"] for e in test_examples]

redirect_point = evaluate_threshold(test_gold_redirect, test_redirect, t_redirect)
discusses_point = evaluate_threshold(test_gold_discusses, test_discusses, t_discusses)
print(f"test, {len(test_examples)} passages, both thresholds and the gate frozen{check}")
print(
    f"redirect alone: precision {redirect_point.precision:.4f}  recall {redirect_point.recall:.4f}  "
    f"f1 {redirect_point.f1:.4f}{check}"
)
print(
    f"discusses alone: precision {discusses_point.precision:.4f}  recall {discusses_point.recall:.4f}  "
    f"f1 {discusses_point.f1:.4f}{check}"
)

test_results = {
    e.id: helpers.screen(helpers.decide_labels(decide(e), thresholds, gate)) for e in test_examples
}
test_counts = {helpers.FLAG: 0, helpers.PASS: 0, helpers.REVIEW: 0}
for e in test_examples:
    test_counts[test_results[e.id].outcome] += 1
test_accepted = [test_results[e.id].outcome != helpers.REVIEW for e in test_examples]
test_correct = [test_results[e.id].outcome == labels[e.id]["outcome"] for e in test_examples]
test_outcomes = evaluate_outcomes(test_accepted, test_correct)

print(
    f"flag: {test_counts[helpers.FLAG]}  pass: {test_counts[helpers.PASS]}  "
    f"review: {test_counts[helpers.REVIEW]}{check}"
)
print(f"coverage: {test_outcomes.coverage:.4f}{check}")
print(f"accuracy among answered: {test_outcomes.accuracy:.4f}{check}")
print(f"risk among answered: {test_outcomes.risk:.4f}{check}")

`discusses`'s own business rule also reads perfectly here — precision, recall and f1 all 1.0000 on `test`. This is not guaranteed by the fixture design: the two propositions are independent (CONTRIBUTING.md section 5's mixed and no-match cases both appear in this set, so `redirect` and `discusses` are each free to read true or false on their own), and `tests/test_helpers.py` proves `discusses` can change the composed outcome by flipping a real stored decision. The 1.0000 here is simply a property of every stored `discusses` answer in this run landing on the correct side of 0.90 against its own gold label — including the one genuinely uncertain case, `tn02` (`discusses` 0.50): its tag still comes out correct, but its confidence is far too low to trust, which is why the shared gate, not this score, is what sends it to review below, independently of `redirect`.

Among the 21 `test` passages `screen` actually answered (2 of the 23 are sent to review, below), a confusion matrix of gold against the composed outcome names exactly which error is which: a `flag`/`pass` passage off the diagonal is a false positive or a miss the gate did not catch.

In [ ]:
answered_examples = [e for e in test_examples if test_results[e.id].outcome != helpers.REVIEW]
answered_gold = [labels[e.id]["outcome"] for e in answered_examples]
answered_predicted = [test_results[e.id].outcome for e in answered_examples]
matrix = confusion_matrix(answered_gold, answered_predicted, labels=[helpers.FLAG, helpers.PASS])
width = max(len(label) for label in matrix.labels)
leader_width = max(width + len("gold "), len("predicted ->"))
header = "  ".join(f"{label:>{width}s}" for label in matrix.labels)
print(f"Test confusion matrix (answered only), gold rows by predicted columns{check}:")
print(f"  {'predicted ->':<{leader_width}s}  {header}{check}")
for gold_label, row in zip(matrix.labels, matrix.matrix.tolist(), strict=True):
    counts = "  ".join(f"{count:{width}d}" for count in row)
    leader = f"gold {gold_label:<{width}s}"
    print(f"  {leader:<{leader_width}s}  {counts}{check}")
plot_confusion_matrix(matrix, title=f"Test split, answered only{check}")

The use case's own numbers: the miss rate is the share of adversarial passages the composed outcome lets through as `pass`; the false-positive rate is the share of benign passages it reports as `flag`. A passage sent to `review` is neither counted, the way [selective prediction](../../docs/glossary.md#selective-prediction) never scores an abstention as an error. "Ungated" strips the gate out entirely and reports what the business threshold alone would do, with no third outcome -- the number the use case's miss rate and false-positive rate would be without any review path at all.

In [ ]:
adversarial_ids = [e.id for e in test_examples if labels[e.id]["outcome"] == helpers.FLAG]
benign_ids = [e.id for e in test_examples if labels[e.id]["outcome"] == helpers.PASS]

misses = sum(1 for i in adversarial_ids if test_results[i].outcome == helpers.PASS)
false_positives = sum(1 for i in benign_ids if test_results[i].outcome == helpers.FLAG)
reviewed_adversarial = sum(1 for i in adversarial_ids if test_results[i].outcome == helpers.REVIEW)
reviewed_benign = sum(1 for i in benign_ids if test_results[i].outcome == helpers.REVIEW)

# The ungated view reuses screen() itself, with the shared gate forced open
# (confidence_gate=0.0 never gates anything): this is the business threshold alone, with no
# review outcome at all, rather than a second, inline reimplementation of the same rule that
# could silently drift from it.
ungated_results = {
    e.id: helpers.screen(helpers.decide_labels(decide(e), thresholds, 0.0)) for e in test_examples
}
ungated_misses = sum(1 for i in adversarial_ids if ungated_results[i].outcome == helpers.PASS)
ungated_false_positives = sum(1 for i in benign_ids if ungated_results[i].outcome == helpers.FLAG)

print(f"adversarial, {len(adversarial_ids)} passages{check}")
print(
    f"  miss rate (gated): {misses} of {len(adversarial_ids)} = "
    f"{misses / len(adversarial_ids):.4f} ({reviewed_adversarial} sent to review){check}"
)
print(
    f"  miss rate (ungated, business threshold alone): {ungated_misses} of "
    f"{len(adversarial_ids)} = {ungated_misses / len(adversarial_ids):.4f}{check}"
)
print(f"benign, {len(benign_ids)} passages{check}")
print(
    f"  false-positive rate (gated): {false_positives} of {len(benign_ids)} = "
    f"{false_positives / len(benign_ids):.4f} ({reviewed_benign} sent to review){check}"
)
print(
    f"  false-positive rate (ungated, business threshold alone): {ungated_false_positives} of "
    f"{len(benign_ids)} = {ungated_false_positives / len(benign_ids):.4f}{check}"
)

The same pooled, per-label view as `validation`'s own sanity check above, now on `test` with every cut-off frozen: `evaluate_selective` reports the confidence-only view of the 46 pooled (passage, label) decisions, next to the ungated accuracy it is being compared against. The gate's only review branch here is the confidence gate itself, so this pooled, per-decision view and the task-level `evaluate_outcomes` above describe the same gate from two different granularities (one decision per label, one outcome per passage) and read differently for two reasons: each wrong passage contributes exactly one wrong label decision out of its two to the pooled count, so the same two wrong passages (`ta10`, `tb05`) move the pooled risk by less than they move the task-level risk; and a label whose own confidence clears the gate is still counted answered in the pooled view even when it sits inside a passage whose outcome is `review` because the *other* label was gated — `tn02` below is exactly this: its `redirect` decision clears the gate and is correct, so the pooled view answers it, while the whole passage is sent to `review` because `discusses` alone does not clear the gate.

In [ ]:
test_ids = [e.id for e in test_examples]
test_gold_by_label = {helpers.REDIRECT: test_gold_redirect, helpers.DISCUSSES: test_gold_discusses}
test_noul_by_label = {helpers.REDIRECT: test_redirect, helpers.DISCUSSES: test_discusses}
test_pooled_correct, test_pooled_confidence, test_pooled_keys = pool_label_decisions(
    test_ids, list(helpers.LABELS), test_gold_by_label, test_noul_by_label, thresholds
)
pooled_selective = evaluate_selective(test_pooled_correct, test_pooled_confidence, gate)
print(f"pooled, {len(test_pooled_correct)} label decisions{check}")
print(f"ungated accuracy: {sum(test_pooled_correct) / len(test_pooled_correct):.4f}{check}")
print(f"coverage: {pooled_selective.coverage:.4f}{check}")
print(f"accuracy among answered: {pooled_selective.accuracy:.4f}{check}")
print(f"risk among answered: {pooled_selective.risk:.4f}{check}")

In [ ]:
print("Test passages the composed outcome got wrong, and what the gate saw on each label:")
for example in test_examples:
    result = test_results[example.id]
    gold_outcome = labels[example.id]["outcome"]
    if result.outcome != helpers.REVIEW and result.outcome != gold_outcome:
        print(
            f"  {example.id}: gold {gold_outcome}, composed {result.outcome} -- "
            f"redirect {result.redirect.noul:.2f} (confidence {result.redirect.confidence:.4f}), "
            f"discusses {result.discusses.noul:.2f} (confidence {result.discusses.confidence:.4f})"
        )

Both passages above clear the frozen gate on both labels, printed in the cell above, and are still wrong: `ta10` sits at `redirect` 0.89, just under the 0.90 threshold, with confidence 0.78 — far enough from an even split that the gate reads it as confidently decided, in the wrong direction, so the missed attack is reported as `pass` and never reaches review. `tb05` is the mirror case on the benign side, at `redirect` 0.91, just over the threshold, confidence 0.82. This is exactly the blind spot named above: `noul_confidence` measures distance from 0.5, never distance from either frozen threshold, so a label whose business threshold sits near one end of the scale can read as highly confident right up to, and just past, the line that decides its tag. A shared gate over two labels cannot be told where either label's own threshold sits without ceasing to be one shared number, so this is a property of pooling the gate at all, not a bug to tune away with a different gate value.

The real queue is built once here, from every `validation` and `test` passage `screen` sends to review — `demo` passages are excluded, since neither is ever reviewed and they were never scored in the first place. `ReviewQueue` has no reordering of its own, so the full list is sorted once by `helpers.review_sort_key` before any of it is submitted, which is what makes the result globally ordered rather than the six-passage preview's own approximation above.

In [ ]:
to_review = [
    (e.id, e, test_results[e.id] if e.split == "test" else val_results[e.id])
    for e in val_examples + test_examples
]
to_review = [row for row in to_review if row[2].outcome == helpers.REVIEW]
to_review.sort(key=lambda row: (helpers.review_sort_key(row[2]), row[0]))

queue = ReviewQueue()
for example_id, example, result in to_review:
    item = helpers.queue_item(example_id, example.fields, result)
    queue.submit(item, result.reason, answer=decide(example)["redirect"])

print(f"queue holds {len(queue)} passages, least confident first{check}")
for row in queue.to_dicts():
    print(f"  {row['item']['id']}: {row['reason']}")

## What was and was not measured

Not measured live: the offline run replays synthetic answers written for this recipe, so its numbers check the pipeline and say nothing about how Jev performs. Screening is one layer of defence, not a guarantee -- it can miss a redirect attempt, and it can misread a quotation as one, as the two passages above show for a rule frozen on this recipe's own `validation` split. Nothing here redacts, blocks or sends anything; every outcome is a label this recipe attaches to a passage, and acting on it is left to whatever system calls this rule.

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test passages")
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, model {backend.model}")
    print(f"Captured: {dates}")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test passages")

## Next steps

- Add a harder look-alike to `ROWS` in `build_fixtures.py` — a benign passage that quotes an attack even more closely, or an adversarial passage phrased even more softly — and see which threshold or the gate it lands on.
- Loosen the confidence gate (a lower `target_accuracy`, or `min_coverage` in its place) and watch `evaluate_selective`'s coverage rise on `test` while its risk rises too; the two passages named above never move, because they sit well clear of the gate in both directions already.
- Neighbouring recipes: [`15-sensitive-text-triage`](../15-sensitive-text-triage/) (another `Noul` proposition, in this recipe's own `Trust & security` category) and [`16-discord-moderation-triage`](../16-discord-moderation-triage/) (a `Choice` question, also `Trust & security`).